In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from helper import *
warnings.filterwarnings("ignore")


In [2]:
accounts_df = load_accounts()
events_df = load_events()
support_tickets_df = load_support_tickets()
feature_usage_df = load_feature_usage()
subscription_df = load_subscriptions()

In [3]:
# Display the first few rows of each DataFrame
print("Accounts DataFrame:")
print(accounts_df.head())
print("\nEvents DataFrame:")
print(events_df.head())
print("\nSupport Tickets DataFrame:")
print(support_tickets_df.head())
print("\nFeature Usage DataFrame:")
print(feature_usage_df.head())
print("\nSubscription DataFrame:")
print(subscription_df.head())

Accounts DataFrame:
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  
0       Basic      9     False       False  
1       Basic     18     False        True  
2       Basic      1     False       False  
3       Basic     24      True       False  
4  Enterprise     35     False        True  

Events DataFrame:
  churn_event_id account_id  churn_date reason_code  refund_amount_usd  \
0       C-816288   A-c37cab  2024-10-27     pricing               4.03   
1       C-5a81e7   A-37f969  2024-06-25     support              96.45   
2     

## Feature Usage Focus Point

In [4]:
feature_usage_df.head()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


In [5]:
feature_usage_df[feature_usage_df.duplicated('usage_id', keep=False)].sort_values('usage_id').head(20)

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


In [6]:
accounts_df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [7]:

# check if the account_id and usage_id have a common pattern that can be used for merging
test_a = accounts_df["account_id"].str.replace(r'^A-','',regex=True)
test_b = feature_usage_df["usage_id"].str.replace(r'^U-','',regex=True)

# check if the cleaned account_id and usage_id have any common values
common_values = set(test_a).intersection(set(test_b))
print(f"Number of common values between cleaned account_id and usage_id: {len(common_values)}")
print(f"Sample common values: {list(common_values)[:10]}")

Number of common values between cleaned account_id and usage_id: 1
Sample common values: ['05d3b3']



After focusing on this table, I can confirm that usage_id identifies who used the feature.The same user can use the product on multiple days, on multiple subscriptions, and use multiple different features. That is not a bug. That is completely normal and expected.
There are no duplicates. There is only a bad column name in the documentation.
```
    19294	U-0c9318	S-01b2dc	2023-12-11	feature_9	5	1060	3	False
    17533	U-0c9318	S-0ffab0	2024-01-30	feature_11	3	1614	0	False
    20588	U-13ce5b	S-9b623b	2023-03-26	feature_28	10	2050	1	False
    9626	U-13ce5b	S-8b0950	2024-09-10	feature_9	8	824	0	True
```
- You can notice here that though the usage_id is the same subscription is not, time is different which means that if a client has multiple subscription and logged from different ones they will be recorded with the same usage_id
The real problem occured on the documentation of the table from the source (Kaggle dataset)

### Recommendation

github copilot comes up with the following recommendation:

- **Do not use `usage_id` alone** as the primary key for `ravenstack_feature_usage` because duplicates exist.
- **Use a composite key (`subscription_id`, `usage_id`)** or add a surrogate `id` column and enforce uniqueness on (`subscription_id`, `usage_id`).

This ensures each row is uniquely identified and preserves the observed relationship where the same `usage_id` can appear under different subscriptions.

### Explanation and Evidence

I ran a quick scan over `raw_data/ravenstack_feature_usage.csv` and found:

- Total rows: 25,000
- Distinct `usage_id` values: 24,979
- Duplicate `usage_id` count: 21 (i.e., 21 `usage_id`s appear more than once)

Examples of duplicated `usage_id`s tied to different `subscription_id`s:

- `U-25b56c` appears with `S-810c27` and `S-34253c`
- `U-662254` appears with `S-3c072b` and `S-f65381`

What I checked and why this supports the recommendation:

- I counted occurrences of `usage_id` and found duplicates (so `usage_id` is not globally unique).
- I tested composite uniqueness: `(subscription_id, usage_id)` is unique across all rows, while `usage_id` alone is not.
- The duplicates correspond to the same user performing usage under different subscriptions or at different times — this matches expected product behavior.

Conclusion: since `usage_id` appears under multiple `subscription_id`s, using `usage_id` alone as the primary key would incorrectly collapse distinct events. Therefore a composite key (`subscription_id`, `usage_id`) (or a surrogate id with a uniqueness constraint on that composite) is the correct design.

This recommendation was produced by github copilot based on the dataset checks above.